<a href="https://colab.research.google.com/github/MarlzRana/machine-learning/blob/main/model_train_segformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Segformer Encoder + UNet Decoder Training

In this notebook we will be training the model that utilizes a SegFormer encoder and UNet decoder

## Imports

External imports

In [ ]:
import torch

from torchvision import transforms

from torch.utils.data import DataLoader

from torch.optim.lr_scheduler import ReduceLROnPlateau

from transformers import SegformerForSemanticSegmentation

Internal imports

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/dataset.ipynb"

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/mod_unet.ipynb"

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/loss.ipynb"

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/epochs.ipynb"

## Constants

In [ ]:
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

TRAIN_IMG_DIR = "drive/MyDrive/CamVid/train"
TRAIN_MASK_DIR = "drive/MyDrive/CamVid/trainannot"

VALID_IMG_DIR = "drive/MyDrive/CamVid/val"
VALID_MASK_DIR = "drive/MyDrive/CamVid/valannot"

## Datasets

Define the image and mask transform

In [ ]:
img_transform = transforms.Compose(
    [
        transforms.Resize(size=(352, 352), antialias=True),
        lambda x: x.to(torch.float) / 255,
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]
)

In [ ]:
mask_transform = transforms.Compose(
    [
        transforms.Resize(size=(352, 352), antialias=True),
    ]
)

Define the label2id map

In [ ]:
label2id = {
        "sky": 0,
        "building": 1,
        "pole": 2,
        "road": 3,
        "pavement": 4,
        "tree": 5,
        "signsymbol": 6,
        "fence": 7,
        "car": 8,
        "pedestrain": 9,
        "bicyclist": 10,
        "unlabelled": 11
}

Define the id2label map

In [ ]:
id2label = {value:key for key, value in label2id.items()}

Define the training datasets

In [ ]:
train_ds_sep_masks = SegmentationDatasetSeperateMasks(
    img_dir=TRAIN_IMG_DIR,
    msk_dir=TRAIN_MASK_DIR,
    label_map=label2id,
    img_transform=img_transform,
    msk_transform=mask_transform,
    msk_type=torch.float
)

NameError: name 'SegmentationDatasetSeperateMasks' is not defined

In [ ]:
train_ds_join_masks = SegmentationDatasetJoinedMsks(
    img_dir=TRAIN_IMG_DIR,
    msk_dir=TRAIN_MASK_DIR,
    img_transform=img_transform,
    msk_type=torch.long,
)

Define the validation dataset

In [ ]:
valid_ds_sep_masks = SegmentationDatasetSeperateMasks(
    img_dir=VALID_IMG_DIR,
    msk_dir=VALID_MASK_DIR,
    label_map=label2id,
    img_transform=img_transform,
    msk_transform=mask_transform,
    msk_type=torch.float,
)

In [ ]:
valid_ds_join_masks = SegmentationDatasetJoinedMsks(
    img_dir=VALID_IMG_DIR,
    msk_dir=VALID_MASK_DIR,
    img_transform=img_transform,
    msk_transform=mask_transform,
    msk_type=torch.long
)

## Models

Create a pretrained SegFormer model

In [ ]:
model_segformer = SegformerForSemanticSegmentation.from_pretrained(
  "nvidia/mit-b5",
  num_labels=len(label2id),
  id2label=id2label,
  label2id=label2id,
  reshape_last_stage=True
).to(DEVICE)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/70.0k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/328M [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b5 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.wei

Create a SegFormer-UNet model that uses a cropping strategy to handle the channel mismatch

In [ ]:
model_segformer_unet_crop = ModUNet(
    num_classes=len(label2id),
    encoder_name="segformer",
    channel_mismatch_strategy="crop"
).to(DEVICE)

Create a SegFormer-UNet model that uses a upconvolution strategy to handle the channel mismatch

In [ ]:
model_segformer_unet_upconv = ModUNet(
    num_classes=len(label2id),
    encoder_name="segformer",
    channel_mismatch_strategy="upconv"
).to(DEVICE)

## Dataloaders

In [ ]:
train_dl_sep_masks = DataLoader(train_ds_sep_masks, batch_size=8, shuffle=True)
valid_dl_sep_masks = DataLoader(valid_ds_sep_masks, batch_size=8, shuffle=True)

In [ ]:
train_dl_join_masks = DataLoader(train_ds_join_masks, batch_size=8, shuffle=True)
valid_dl_join_masks = DataLoader(valid_ds_join_masks, batch_size=8, shuffle=True)

## Loss

In [ ]:
loss_fn = DiceLoss()

## Optimizers

In [ ]:
optim_segformer = torch.optim.Adam(
    model_segformer.parameters(),
    lr=1e-5
)

In [ ]:
optim_segformer_unet_crop = torch.optim.Adam(
    model_segformer_unet_crop.parameters(),
    lr=1e-4
)

In [ ]:
optim_segformer_unet_upconv = torch.optim.Adam(
    model_segformer_unet_upconv.parameters(),
    lr=1e-4
)

## Learning Rate Scheduler

In [ ]:
lr_scheduler_segformer = ReduceLROnPlateau(optim_segformer, mode='min', patience=5, factor=1e-1, verbose=True)

In [ ]:
lr_scheduler_segformer_unet_crop = ReduceLROnPlateau(optim_segformer_unet_crop, mode='min', patience=5, factor=1e-1, verbose=True)

In [ ]:
lr_scheduler_segformer_unet_upconv = ReduceLROnPlateau(optim_segformer_unet_upconv, mode='min', patience=5, factor=1e-1, verbose=True)

## Train

Train/finetune the pretrained SegFormer

In [ ]:
train_epoch_segformer = SegFormerTrainEpoch(
    dataloader=train_dl_join_masks,
    model=model_segformer,
    optimizer=optim_segformer,
    metrics=[],
    device=DEVICE
)

valid_epoch_segformer = SegFormerEvalEpoch(
    dataloader=valid_dl_join_masks,
    model=model_segformer,
    metrics=[],
    device=DEVICE
)

In [ ]:
NUM_EPOCHS = 10

for i in range(NUM_EPOCHS):
  train_loss = train_epoch_segformer.run()
  val_loss = valid_epoch_segformer.run()
  lr_scheduler_segformer.step(val_loss)

 39%|███▉      | 144/367 [01:28<02:03,  1.81it/s, train_loss=2.18]

In [ ]:
torch.save(model_segformer, "./drive/MyDrive/checkpoints/model_segformer.pth")

Train the SegFormer-UNet model that uses a cropping strategy to handle the channel mismatch

In [ ]:
train_epoch_segformer_unet_crop = TrainEpoch(
    dataloader=train_dataloader,
    model=model_segformer_unet_crop,
    optimizer=optimizer_segformer_unet_crop,
    loss_fn=loss_fn,
    metrics=[],
    device=DEVICE
)

valid_epoch_segformer_unet_crop = EvalEpoch(
    dataloader=valid_dataloader,
    model=model_segformer_unet_crop,
    loss_fn=loss_fn,
    metrics=[],
    device=DEVICE
)

In [ ]:
NUM_EPOCHS = 40

for i in range(NUM_EPOCHS):
  train_loss = train_epoch_segformer_unet_crop.run()
  val_loss = valid_epoch_segformer_unet_crop.run()
  lr_scheduler_segformer_unet_crop.step(val_loss)

100%|██████████| 101/101 [00:04<00:00, 21.17it/s, eval_loss=0.125]


In [ ]:
torch.save(model_segformer_unet_crop, "./drive/MyDrive/checkpoints/model_segformer_unet_crop.pth")

Train the SegFormer-UNet model that uses a upconv strategy to handle the channel mismatch

In [ ]:
train_epoch_segformer_unet_upconv = TrainEpoch(
    dataloader=train_dataloader,
    model=model_segformer_unet_upconv,
    optimizer=optimizer_segformer_unet_upconv,
    loss_fn=loss_fn,
    metrics=[],
    device=DEVICE
)

valid_epoch_segformer_unet_upconv = EvalEpoch(
    dataloader=valid_dataloader,
    model=model_segformer_unet_upconv,
    loss_fn=loss_fn,
    metrics=[],
    device=DEVICE
)

In [ ]:
NUM_EPOCHS = 40

for i in range(NUM_EPOCHS):
  print(f"Epoch {i + 1}:")
  train_loss = train_epoch_segformer_unet_upconv.run()
  val_loss = valid_epoch_segformer_unet_upconv.run()
  lr_scheduler_segformer_unet_upconv.step(val_loss)

Epoch 1:


100%|██████████| 101/101 [00:05<00:00, 18.39it/s, eval_loss=0.75]


Epoch 2:


100%|██████████| 101/101 [00:05<00:00, 17.61it/s, eval_loss=0.714]


Epoch 3:


...

100%|██████████| 101/101 [00:06<00:00, 16.50it/s, eval_loss=0.142]


Epoch 39:


100%|██████████| 101/101 [00:04<00:00, 21.17it/s, eval_loss=0.143]


Epoch 40:


100%|██████████| 101/101 [00:05<00:00, 18.54it/s, eval_loss=0.141]


In [ ]:
torch.save(model_segformer_unet_upconv, "./drive/MyDrive/checkpoints/model_segformer_unet_upconv.pth")